In [ ]:
%maven org.knowm.xchart:xchart:3.8.8
// Setup 1/2: fetch the plotting library

In [ ]:
// ==== Setup 2/2: imports and chart helpers ====

import org.knowm.xchart.*;
import org.knowm.xchart.style.Styler;
import java.awt.image.BufferedImage;
import java.util.Random;

// chart helpers - not part of the lesson
BufferedImage one(String title, double[] xs, double[] ys) {
    XYChart ch = new XYChartBuilder().width(400).height(300)
            .title(title).xAxisTitle("N").yAxisTitle("comparisons").build();
    ch.getStyler().setLegendVisible(false);
    ch.getStyler().setMarkerSize(4);
    ch.addSeries(title, xs, ys);
    return BitmapEncoder.getBufferedImage(ch);
}

BufferedImage row(BufferedImage... imgs) {
    int w = 0, h = 0;
    for (BufferedImage i : imgs) { w += i.getWidth(); h = Math.max(h, i.getHeight()); }
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    java.awt.Graphics g = out.getGraphics();
    g.setColor(java.awt.Color.WHITE); g.fillRect(0, 0, w, h);
    int x = 0;
    for (BufferedImage i : imgs) { g.drawImage(i, x, 0, null); x += i.getWidth(); }
    g.dispose();
    return out;
}

BufferedImage plot(String title, String yLabel, double[] xs, String[] names, double[][] ys) {
    XYChart ch = new XYChartBuilder().width(700).height(400)
            .title(title).xAxisTitle("N").yAxisTitle(yLabel).build();
    ch.getStyler().setLegendPosition(Styler.LegendPosition.InsideNW);
    ch.getStyler().setMarkerSize(5);
    for (int i = 0; i < names.length; i++) ch.addSeries(names[i], xs, ys[i]);
    return BitmapEncoder.getBufferedImage(ch);
}

int comparisons;    // incremented once per element inspected

# Binary search and recursion

## Phone book: 1000 pages, one name

1. **Linear** - page 1, then 2, then 3, ...
2. **Jump** - 10 at a time, back up when past it
3. **Binary** - open the middle, discard half

2 feels much better than 1. Is it?

## Linear search

No assumptions. Look at everything.

In [ ]:
int linearSearch(int[] a, int key) {
    for (int i = 0; i < a.length; i++) {
        comparisons++;
        if (a[i] == key) return i;
    }
    return -1;
}

int[] a = new int[1000];
for (int i = 0; i < a.length; i++) a[i] = 2 * i;

comparisons = 0; linearSearch(a, 0);
System.out.println("first  : " + comparisons);
comparisons = 0; linearSearch(a, 1998);
System.out.println("last   : " + comparisons);
comparisons = 0; linearSearch(a, 1999);
System.out.println("absent : " + comparisons);

## Jump search

Sorted. Jump 10, overshoot, back up.

In [ ]:
int STEP = 10;

int jumpSearch(int[] a, int key) {
    int i = 0;
    while (i < a.length) {                 // jump
        comparisons++;
        if (a[i] >= key) break;
        i += STEP;
    }
    int from = Math.max(0, i - STEP);      // walk the skipped block
    int to   = Math.min(i, a.length - 1);
    for (int j = from; j <= to; j++) {
        comparisons++;
        if (a[j] == key) return j;
    }
    return -1;
}

comparisons = 0; jumpSearch(a, 1998);
System.out.println("jump   : " + comparisons);
comparisons = 0; linearSearch(a, 1998);
System.out.println("linear : " + comparisons);

10x fewer. Now grow N.

In [ ]:
System.out.printf("%10s %10s %10s %10s %10s%n", "N", "linear", "jump", "lin ratio", "jmp ratio");
double prevL = 0, prevJ = 0;
for (int n = 1000; n <= 32000; n *= 2) {
    int[] arr = new int[n];
    for (int i = 0; i < n; i++) arr[i] = 2 * i;

    comparisons = 0; linearSearch(arr, 2 * (n - 1)); double cl = comparisons;
    comparisons = 0; jumpSearch(arr, 2 * (n - 1));   double cj = comparisons;

    System.out.printf("%10d %10.0f %10.0f %10s %10s%n", n, cl, cj,
        prevL == 0 ? "-" : String.format("%.2f", cl / prevL),
        prevJ == 0 ? "-" : String.format("%.2f", cj / prevJ));
    prevL = cl; prevJ = cj;
}

Read the ratio columns, not the counts.

**Same ratio: N doubles => operations double.**

Jump = linear with a better constant. Shape unchanged.

## Binary search

Open the middle, discard half.

`left..right` = the only region that can still hold the key.

In [ ]:
int binarySearch(int[] a, int key) {
    int left = 0, right = a.length - 1;

    while (left <= right) {                     // <= : a region of one still counts
        int mid = (left + right) / 2;
        comparisons++;
        if      (a[mid] < key) left  = mid + 1;  // mid just tested
        else if (a[mid] > key) right = mid - 1;
        else                   return mid;
    }
    return -1;
}

comparisons = 0;
System.out.println("index " + binarySearch(a, 1234) + ", " + comparisons + " comparisons");

In [ ]:
int left = 0, right = a.length - 1, key = 1234;
while (left <= right) {
    int mid = (left + right) / 2;
    System.out.printf("left=%-5d right=%-5d mid=%-5d   region %d%n",
                      left, right, mid, right - left + 1);
    if      (a[mid] < key) left  = mid + 1;
    else if (a[mid] > key) right = mid - 1;
    else break;
}

1000 -> 500 -> 250 -> 125 -> ... 10 steps.

## Testing on random data

Random sorted arrays, random keys, half present / half absent.

Does this tell us anything about worst case?

What would change if we assume that the key is always present?

Does this *prove* anything?

In [ ]:
Random rng = new Random(42);

double[] measure(int n, int trials) {
    int[] arr = new int[n];
    int v = 0;
    for (int i = 0; i < n; i++) { v += 1 + rng.nextInt(5); arr[i] = v; }

    long sl = 0, sj = 0, sb = 0;
    for (int t = 0; t < trials; t++) {
        int k = (t % 2 == 0) ? arr[rng.nextInt(n)] : arr[rng.nextInt(n)] - 1;
        comparisons = 0; linearSearch(arr, k); sl += comparisons;
        comparisons = 0; jumpSearch(arr, k);   sj += comparisons;
        comparisons = 0; binarySearch(arr, k); sb += comparisons;
    }
    return new double[] { (double) sl / trials, (double) sj / trials, (double) sb / trials };
}

System.out.printf("%10s %10s %10s %10s%n", "N", "linear", "jump", "binary");
for (int n = 2000; n <= 32000; n *= 2) {
    double[] r = measure(n, 400);
    System.out.printf("%10d %10.1f %10.1f %10.1f%n", n, r[0], r[1], r[2]);
}

We run each algorithm on random data 200 times and collect the operation count, then plot each one on its own y scale.

**We are NOT measuring runtime!**

In [ ]:
int points = 12;
double[] ns = new double[points];
double[][] ys = new double[3][points];

int n = 1000;
for (int i = 0; i < points; i++) {
    double[] r = measure(n, 200);
    ns[i] = n;
    ys[0][i] = r[0]; ys[1][i] = r[1]; ys[2][i] = r[2];
    n *= 2;
}

row(one("Linear search", ns, ys[0]),
    one("Jump search", ns, ys[1]),
    one("Binary search", ns, ys[2]))

See how the operation count ratio increases when going from N to 2N:

In [ ]:
double[][] ratios = new double[3][points - 1];
double[] ns2 = new double[points - 1];
for (int i = 1; i < points; i++) {
    ns2[i - 1] = ns[i];
    for (int k = 0; k < 3; k++) ratios[k][i - 1] = ys[k][i] / ys[k][i - 1];
}
plot("Op count multiplier when N doubles", "ops(2N) / ops(N)", ns2,
     new String[] { "linear", "jump", "binary" }, ratios)

## Why log N

Number of "candidates" at each step:
```
N -> N/2 -> N/4 -> ... -> 1
```

count steps:
```
N / 2^k = 1   =>   2^k = N   =>   k = log2 N
```

| N | linear | binary |
|---|---|---|
| 10^3 | 1 000 | 10 |
| 10^6 | 1 000 000 | 20 |
| 10^9 | 1 000 000 000 | 30 |

N doubles => **+1** iteration.

### Why the base does not matter

What if we threw away 9/10 of the array each step instead of 1/2?

In [ ]:
// how many times can we divide N by b before reaching 1?
int steps(double N, int b) {
    int k = 0;
    while (N > 1) { N = N / b; k++; }
    return k;
}

System.out.printf("%14s %8s %8s %8s %14s%n", "N", "b = 2", "b = 4", "b = 10", "b=2 / b=10");
for (long n = 1000; n <= 1_000_000_000L; n *= 10) {
    int s2 = steps(n, 2), s4 = steps(n, 4), s10 = steps(n, 10);
    System.out.printf("%14d %8d %8d %8d %14.2f%n", n, s2, s4, s10, (double) s2 / s10);
}

System.out.println("\nlog2(10) = " + (Math.log(10) / Math.log(2)));

```
log_b N = log_2 N / log_2 b
```

### Does the constant ever matter?

A linear algorithm with a **big** constant against a quadratic one with a **small** constant.
Which one would you use?

In [ ]:
int pts = 40;
double[] xs2 = new double[pts];
double[][] cc = new double[2][pts];
for (int i = 0; i < pts; i++) {
    double N = (i + 1) * 50;        // 50, 100, ... 2000
    xs2[i]   = N;
    cc[0][i] = 100 * N;             // linear,    huge constant
    cc[1][i] = N * N / 10;          // quadratic, tiny constant
}

System.out.printf("%8s %14s %14s   %s%n", "N", "100 N", "N^2 / 10", "winner");
for (int i = 3; i < pts; i += 6) {
    System.out.printf("%8.0f %14.0f %14.0f   %s%n", xs2[i], cc[0][i], cc[1][i],
                      cc[0][i] < cc[1][i] ? "linear" : "quadratic");
}

plot("100 N  vs  N^2 / 10", "operations", xs2,
     new String[] { "100 N  (linear)", "N^2 / 10  (quadratic)" }, cc)

## Common growth classes

Here we are just plotting functions, not running anything.

In [ ]:
// one definition of each class, used for BOTH the plot and the table below
String[] cls = { "O(1)", "O(log N)", "O(N)", "O(N log N)", "O(N^2)", "O(N^3)", "O(2^N)" };

double ops(int which, double N) {
    switch (which) {
        case 0:  return 1;                             // constant
        case 1:  return Math.log(N) / Math.log(2);     // logarithmic
        case 2:  return N;                             // linear
        case 3:  return N * Math.log(N) / Math.log(2); // linearithmic
        case 4:  return N * N;                         // quadratic
        case 5:  return N * N * N;                     // cubic
        default: return Math.pow(2, N);                // exponential
    }
}

int m = 40;
double[] xs = new double[m];
double[][] g = new double[cls.length][m];
for (int i = 0; i < m; i++) {
    double N = i + 1;
    xs[i] = N;
    for (int k = 0; k < cls.length; k++) g[k][i] = ops(k, N);
}

XYChart ch = new XYChartBuilder().width(700).height(450)
        .title("Growth classes").xAxisTitle("N").yAxisTitle("operations").build();
ch.getStyler().setLegendPosition(Styler.LegendPosition.InsideNE);
ch.getStyler().setMarkerSize(0);
ch.getStyler().setYAxisMax(1400.0);            // 2^N leaves the chart almost immediately
ch.getStyler().setXAxisMax(40.0);
for (int i = 0; i < cls.length; i++) ch.addSeries(cls[i], xs, g[i]);

String[] name = { "constant", "logarithmic", "linear", "linearithmic",
                  "quadratic", "cubic", "exponential" };
String[] eg   = { "fixed no. of operations", "binary search", "linear search", "sorting",
                  "every pair", "every triple", "all subsets" };

String fmt(double v) {
    if (Double.isInfinite(v)) return "too big";
    if (v < 1e5) return String.format("%,.0f", v);
    return String.format("%.0e", v);
}

double[] sizes = { 10, 1_000, 1_000_000 };

System.out.printf("%-11s %-13s %-18s", "class", "name", "example");
for (double N : sizes) System.out.printf("%14s", "N = " + fmt(N));
System.out.println();
for (int k = 0; k < cls.length; k++) {
    System.out.printf("%-11s %-13s %-18s", cls[k], name[k], eg[k]);
    for (double N : sizes) System.out.printf("%14s", fmt(ops(k, N)));
    System.out.println();
}

BitmapEncoder.getBufferedImage(ch)

### Cost of binary search on non-sorted data

Binary search needs sorted data. Sorting done properly (next week!) is O(N log N).
When would you use it?

---

# Counting operations: practice

In [ ]:
// N values we test every example on
int[] NS = { 250, 500, 1000, 2000, 4000 };

// prints a table and the multiplier from one row to the next
void growth(String label, long[] counts) {
    System.out.println(label);
    System.out.printf("  %8s %14s %8s%n", "N", "operations", "ratio");
    for (int i = 0; i < NS.length; i++) {
        String ratio = (i == 0) ? "-" : String.format("%.2f", (double) counts[i] / counts[i - 1]);
        System.out.printf("  %8d %14d %8s%n", NS[i], counts[i], ratio);
    }
}

### 1. Every unordered pair

Inner loop runs n-1, then n-2, then n-3, ...

In [ ]:
int everyPair(int n) {
    int ops = 0;
    for (int i = 0; i < n; i++)
        for (int j = i + 1; j < n; j++)
            ops++;
    return ops;
}

long[] counts = new long[NS.length];
for (int i = 0; i < NS.length; i++) counts[i] = everyPair(NS[i]);
growth("Every pair", counts);

System.out.println("\n  n(n-1)/2 at n=4000 : " + (4000L * 3999 / 2));

Half the work of the full double loop. Ratio still **4** when doubling.

### 2. Sequential vs nested

In [ ]:
int sequential(int n) {
    int ops = 0;
    for (int i = 0; i < n; i++) ops++;
    for (int j = 0; j < n; j++) ops++;
    return ops;
}

int nested(int n) {
    int ops = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            ops++;
    return ops;
}

for (int i = 0; i < NS.length; i++) counts[i] = sequential(NS[i]);
growth("Sequential - ADD", counts);

System.out.println();
for (int i = 0; i < NS.length; i++) counts[i] = nested(NS[i]);
growth("Nested - MULTIPLY", counts);

### 3. Not what they look like

In [ ]:
int nested100(int n) {         // inner bound is not n
    int ops = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < 100; j++)
            ops++;
    return ops;
}

int halving(int n) {                   // divided, not incremented
    int ops = 0;
    for (int i = n; i > 1; i = i / 2)
        ops++;
    return ops;
}

for (int i = 0; i < NS.length; i++) counts[i] = nested100(NS[i]);
growth("Nested, inner bound 100", counts);

System.out.println();
for (int i = 0; i < NS.length; i++) counts[i] = halving(NS[i]);
growth("Halving", counts);

Nested + constant inner = **linear**. Nested does not mean squared.

Halving = **+1** per doubling. Same shape as binary search.

### 4. Hidden loops

In [ ]:
int searchInALoop(int n) {
    int[] arr = new int[n];
    for (int i = 0; i < n; i++) arr[i] = 2 * i;

    comparisons = 0;
    for (int i = 0; i < n; i++)
        linearSearch(arr, -1);          // absent: scans the whole array
    return comparisons;
}

for (int i = 0; i < NS.length; i++) counts[i] = searchInALoop(NS[i]);
growth("linearSearch inside a loop", counts);

In [ ]:
int charsCopied(int n) {
    int copied = 0;
    String result = "";
    for (int i = 0; i < n; i++) {
        copied += result.length();
        result += "a";
    }
    return copied;
}

for (int i = 0; i < NS.length; i++) counts[i] = charsCopied(NS[i]);
growth("result += \"a\"  (chars copied)", counts);
// strings are immutable: every += copies everything so far

1 + 2 + ... + N chars copied. Same count as "every pair".

This is the `repeatstring` demo I showed in the first class.

In [ ]:
for (int n = 20000; n <= 160000; n *= 2) {
    long t0 = System.nanoTime();
    String s = "";
    for (int i = 0; i < n; i++) s += "a";
    double concat = (System.nanoTime() - t0) / 1e6;

    t0 = System.nanoTime();
    StringBuilder sb = new StringBuilder();
    for (int i = 0; i < n; i++) sb.append('a');
    double builder = (System.nanoTime() - t0) / 1e6;

    System.out.printf("n=%-8d  +=  %8.1f ms    StringBuilder %6.2f ms%n", n, concat, builder);
}

### 5. Not everything is in terms of N

Max of the minima of two **unsorted** arrays.

In [ ]:
int ops;

int maxOfMinima(int[] x, int[] y) {
    int minX = Integer.MAX_VALUE;
    for (int v : x) { ops++; if (v < minX) minX = v; }   // must see all of x

    int minY = Integer.MAX_VALUE;
    for (int v : y) { ops++; if (v < minY) minY = v; }   // must see all of y

    return Math.max(minX, minY);
}

System.out.printf("%8s %8s %10s%n", "n", "m", "operations");
int[][] sizes = { {1000, 1000}, {1000, 2000}, {2000, 1000}, {1000, 10000}, {10000, 10} };
for (int[] s : sizes) {
    ops = 0;
    maxOfMinima(new int[s[0]], new int[s[1]]);
    System.out.printf("%8d %8d %10d%n", s[0], s[1], ops);
}

Two inputs, two independent sizes.

**O(n + m)**, not O(N). There is no single N to double.

Last row: m constant => **O(n)**.

**Always say what your variable counts (unless obvious).**

---

# Recursion

A programming technique that reduces a problem instance to smaller instances of the **same problem**, until all the small problems are **trivial** to solve.

- **Base case** - answer immediately
- **Recursive case** - shrink, then combine what comes back

In [ ]:
long factorial(int n) {
    if (n == 0) return 1;            // base case
    return n * factorial(n - 1);     // recursive case
}

factorial(10)

In [ ]:
long factorialTraced(int n, int depth) {
    String pad = "  ".repeat(depth);
    System.out.println(pad + "-> factorial(" + n + ")");
    if (n == 0) {
        System.out.println(pad + "<- 1   (base case)");
        return 1;
    }
    long result = n * factorialTraced(n - 1, depth + 1);
    System.out.println(pad + "<- " + result);
    return result;
}

factorialTraced(4, 0);

5 calls down before 1 comes back.

Each call has its own `n`.

In [ ]:
try {
    factorial(-1);                   // base case never reached
} catch (StackOverflowError e) {
    System.out.println("StackOverflowError");
}

## Writing a recursive method
1. **Base case**: What instances of the problem are trivial to solve?
2. **Recursive call**: Given a generic instance, how can I make it simpler, or "closer" to one of the trivial instances?
3. **Combine the results**: How can I use the answer of the simpler instances to solve the harder one?
4. **Check for termination**: Does every path in my algorithm eventually reach a base case?

## Reverse a string

Each version broken by an input the last one missed.

In [ ]:
// v1 - base case: one character
String reverse1(String str) {
    if (str.length() == 1) return str;
    int last = str.length() - 1;
    return str.charAt(last) + reverse1(str.substring(0, last));
}

System.out.println(reverse1("moon"));
try { reverse1(""); } catch (Exception e) { System.out.println("on \"\": " + e); }

In [ ]:
// v2 - base case: empty string
String reverse2(String str) {
    if (str.length() == 0) return str;
    int last = str.length() - 1;
    return str.charAt(last) + reverse2(str.substring(0, last));
}

System.out.println("on \"\": [" + reverse2("") + "]");
try { reverse2(null); } catch (Exception e) { System.out.println("on null: " + e); }

In [ ]:
// v3 - null handled, but checked on EVERY call
String reverse3(String str) {
    if (str == null) return null;
    if (str.length() == 0) return str;
    int last = str.length() - 1;
    return str.charAt(last) + reverse3(str.substring(0, last));
}

System.out.println(reverse3(null) + ", " + reverse3("moon"));

In [ ]:
// v4 - wrapper
public String reverse(String str) {
    if (str == null) return null;
    return reverseRec(str);
}

private String reverseRec(String str) {
    if (str.length() == 0) return str;
    int last = str.length() - 1;
    return str.charAt(last) + reverseRec(str.substring(0, last));
}

System.out.println(reverse(null) + ", " + reverse("moon"));

Recursive methods often have a **wrapper**:
- public wrapper validates input and starts the recursion;
- private recursive method trusts the wrapper.

## Binary search, recursive

```
while (left <= right)   ->   if (left > right) return -1;
left = mid + 1          ->   binSearchRec(a, key, mid + 1, right)
```

In [ ]:
public int binarySearch2(int[] a, int key) {   // wrapper
    return binSearchRec(a, key, 0, a.length - 1);
}

private int binSearchRec(int[] a, int key, int left, int right) {
    if (left > right) return -1;                          // base case: empty
    int mid = (left + right) / 2;
    if      (a[mid] < key) return binSearchRec(a, key, mid + 1, right);
    else if (a[mid] > key) return binSearchRec(a, key, left, mid - 1);
    else                   return mid;                    // base case: found
}

System.out.println(binarySearch2(a, 1234) + ", " + binarySearch2(a, 1235));

In [ ]:
int depth(int n) { return n == 0 ? 0 : 1 + depth(n / 2); }

System.out.println("N = 10^9 -> " + depth(1_000_000_000) + " frames");
System.out.println("factorial(10^6) -> 10^6 frames, dies");

When to use recursion vs. iteration?
Trace the function below:

In [ ]:
long foo(long a, long b) {
    if (a == 0) return 1;
    long val = foo(a / 2, b);
    if (a % 2 == 0) return val * val;
    else            return val * val * b;
}

int power(int a, int b) {
    if (a == 0) return 1;
    return b * power(a - 1, b);
}

System.out.println(foo(100000, 2));
System.out.println(power(100000, 2));

# A couple of exercises

### Contains

Recursively: does `s` contain `c`? Then the same for an `int[]` - that is linear search, recursive.

In [ ]:
boolean contains(String s, char c) {
    return false;   // TODO
}

### Palindrome

Same forwards and backwards. Then ignore spaces and punctuation, so `"A nut for a jar of tuna."` counts.
No loops.

In [ ]:
boolean isPalindrome(String s) {
    return false;   // TODO
}